In [3]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
import os
from pathlib import Path

model_name = "xlm-roberta-base"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
except Exception as e:
    raise RuntimeError(f"Failed to load model '{model_name}': {e}")

# --- Modification to load filtered.json ---
json_path = Path("/content/filtered.json")

if not json_path.exists():
    raise FileNotFoundError(f"Couldn't find '{json_path}'")

# load into pandas
df = pd.read_json(json_path)

print(f"Dataset chargé: {len(df)} lignes — fichier: {json_path}")
print("Premières 5 lignes du dataset:")
display(df.head(5))

# Extract text column (allow common alternative column names)
text_col = None
for c in ["text", "sentence", "sent", "original_text", "idiom_text", "english_text", "chinese_text", "chinese"]:
    if c in df.columns:
        text_col = c
        break
if text_col is None:
    raise KeyError("Le fichier doit contenir une colonne 'text' (ou 'sentence','sent','original_text','idiom_text','english_text','chinese_text','chinese').")
if text_col != "text":
    df = df.rename(columns={text_col: "text"})
sentences = df["text"].astype(str).tolist()
contains_idioms = df["contains_idioms"].tolist() if "contains_idioms" in df.columns else [False] * len(sentences)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dataset chargé: 4310 lignes — fichier: /content/filtered.json
Premières 5 lignes du dataset:


,id,chinese,gold,human,machine
0,0,一波未平，一波又起,suffer a string of reverses,[hardly has one wave subsided when another ris...,"[One wave is not flat, another wave is rising,..."
1,1,一败涂地,suffer a crushing defeat,"[be ruined completely, utter failure, a comple...","[A crushing defeat, a total failure, A defeat]"
2,2,一般见识,lower oneself to the same level as somebody else,[],"[General knowledge, General Insight]"
3,3,一板三眼,following a prescribed pattern in speech or ac...,"[scrupulous and methodical, very careful of th...","[One board and three eyes, A board with three ..."
4,4,一本正经,put on a solemn look,"[assume a dead serious attitude, with feigned ...","[serious, in earnest, in good taste, in a seri..."


In [4]:
# Générer les embeddings (batched, GPU, mixed precision)
import math
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

use_fp16 = device.type == "cuda"

if use_fp16:
    model.half()  # reduce memory & speed up on GPU

def mean_pooling(outputs, attention_mask):
    token_embeddings = outputs.last_hidden_state  # (batch, seq_len, dim)
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

batch_size = 64  # try 32/64/128 depending on GPU memory
n = len(sentences)
embeddings_batches = []

print("Génération des embeddings (batched)...")
for start in tqdm(range(0, n, batch_size)):
    batch = sentences[start : start + batch_size]
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        if use_fp16:
            with torch.cuda.amp.autocast():
                outputs = model(**inputs)
        else:
            outputs = model(**inputs)

        batch_emb = mean_pooling(outputs, inputs["attention_mask"])  # (batch, dim)
        embeddings_batches.append(batch_emb.cpu().numpy())

embeddings = np.vstack(embeddings_batches)



Génération des embeddings (batched)...


100%|██████████| 68/68 [01:50<00:00,  1.63s/it]


In [6]:
import pandas as pd
from umap import UMAP
from sklearn.neighbors import NearestNeighbors

# Using the 'df' and 'embeddings' already generated from 'filtered.json'

reducer = UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
projection = reducer.fit_transform(embeddings)

df['projection_x'] = projection[:, 0]
df['projection_y'] = projection[:, 1]

nn = NearestNeighbors(n_neighbors=16, metric='cosine')
nn.fit(embeddings)
distances, indices = nn.kneighbors(embeddings)

df['neighbors'] = [indices[i][1:].tolist() for i in range(len(df))]

# Sauvegarder tout
df.to_parquet('embeddings_ready.parquet')
print("Données sauvegardées dans embeddings_ready.parquet")

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Données sauvegardées dans embeddings_ready.parquet


In [7]:
import plotly.express as px

# Create an interactive scatter plot of the UMAP projection
fig = px.scatter(
    df,
    x='projection_x',
    y='projection_y',
    hover_name='text', # Display idiom text on hover
    color='contains_idioms' if 'contains_idioms' in df.columns else None, # Color by 'contains_idioms' if available
    title='UMAP Projection of Idioms',
    labels={'projection_x': 'UMAP Dimension 1', 'projection_y': 'UMAP Dimension 2'}
)

fig.show()

### Résumé de l'analyse des idiomes chinois

Ce notebook effectue une analyse de similarité sur un ensemble d'idiomes chinois. Les étapes clés sont les suivantes :

1.  **Chargement des données** : Le fichier `filtered.json` contenant les idiomes (dans la colonne 'chinese') a été chargé dans un DataFrame pandas.
2.  **Génération d'embeddings** : Le modèle pré-entraîné `xlm-roberta-base` a été utilisé pour transformer chaque idiome en un vecteur numérique (embedding), capturant ainsi leur signification sémantique.
3.  **Réduction de dimensionnalité (UMAP)** : Afin de visualiser ces embeddings de haute dimension, l'algorithme UMAP (Uniform Manifold Approximation and Projection) a été appliqué pour projeter les données dans un espace bidimensionnel (projection_x, projection_y).
4.  **Détection des voisins proches** : La méthode des K plus proches voisins (Nearest Neighbors) a été utilisée pour identifier les idiomes les plus similaires pour chaque entrée, basés sur la similarité cosinus de leurs embeddings.
5.  **Visualisation interactive** : Un graphique interactif a été généré avec Plotly Express, permettant d'explorer la distribution des idiomes dans l'espace UMAP et d'identifier visuellement les groupes d'idiomes sémantiquement proches.